<a href="https://colab.research.google.com/github/Salmonyeonwoo/Portfolio-for-application-of-AI-Job-offers/blob/main/crewAI_ipynb%EC%9D%98_%EB%8B%A4%EB%A5%B8_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 종합 투자 리서치팀 - CrewAI + Serper API 프로젝트

## Level 3: 고급 응용 프로젝트 (Serper API 버전)

### 투자 면책 조항
**이 코드는 교육 목적으로만 제작되었습니다.**
- 실제 투자 결정에 사용하지 마시고, 모든 투자는 본인의 책임 하에 진행하시기 바랍니다.
- AI가 생성한 분석은 참고용일 뿐이며, 전문적인 투자 조언을 대체할 수 없습니다.

### 프로젝트 개요
이 프로젝트는 CrewAI와 Serper API를 활용하여 다음과 같은 5명의 전문가 AI 에이전트로 구성된 투자 리서치팀을 구축합니다:

1. **데이터 수집가**: Serper API를 통한 주식 데이터 및 재무 정보 수집
2. **뉴스 분석가**: 최신 뉴스 수집 및 감정 분석
3. **기술적 분석가**: 수집된 데이터로 차트 패턴 및 기술적 지표 분석  
4. **기본 분석가**: 재무제표 및 기업 가치 분석
5. **보고서 작성자**: 종합 분석 결과 정리

### Serper API 활용
- 실시간 주가 데이터 검색
- 재무제표 정보 수집
- 최신 뉴스 및 분석 자료 확보
- 경쟁사 비교 데이터 수집


## 1. 필수 라이브러리 설치

먼저 필요한 패키지들을 설치해야 합니다. 터미널에서 다음 명령어를 실행하거나, 아래 셀의 주석을 해제하여 실행하세요.

```bash
pip install crewai crewai-tools pandas matplotlib seaborn numpy requests
```

### 라이브러리 기능 설명
- **crewai**: 다중 에이전트 AI 프레임워크
- **crewai-tools**: CrewAI용 추가 도구들 (SerperDevTool 포함)
- **pandas, numpy**: 데이터 분석 및 처리
- **matplotlib, seaborn**: 데이터 시각화
- **requests**: HTTP 요청 (추가 API 호출용)


In [ ]:
# 필요한 경우 패키지 설치 (주석 해제하여 실행)
!pip install crewai crewai-tools -q

In [ ]:
# 필수 라이브러리 임포트
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# CrewAI 라이브러리
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, FileWriterTool

## 2. API 키 설정

이 프로젝트를 실행하기 위해서는 다음 API 키들이 **모두 필요**합니다:

### 필수 API 키들
1. **OpenAI API Key**: AI 에이전트들이 사용할 LLM (GPT-4o-mini)
   - 발급: https://platform.openai.com/api-keys
   - 예상 비용: 분석당 약 $0.50 - $2.00

2. **Serper API Key**: 주식 데이터 및 뉴스 검색용 (필수)
   - 발급: https://serper.dev (무료 월 2,500회)
   - 이 프로젝트에서는 주식 데이터 수집의 핵심 도구

### .env 파일 생성 방법
프로젝트 폴더에 `.env` 파일을 생성하고 다음 내용을 추가하세요:

```
OPENAI_API_KEY=your_openai_api_key_here
SERPER_API_KEY=your_serper_api_key_here
```

### Serper API 활용 범위
- 실시간 주가 데이터 검색
- 재무제표 및 기업 정보 수집
- 최신 뉴스 및 시장 분석
- 경쟁사 및 업계 동향 파악


In [ ]:
# API 키 설정 및 확인
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
SERPER_API_KEY = userdata.get('SERPER_API_KEY')

# API 키가 없는 경우 직접 입력 (보안 주의)
# if not OPENAI_API_KEY:
    # OPENAI_API_KEY = "your_key_here"  # 직접 입력시 주석 해제

# if not SERPER_API_KEY:
    # SERPER_API_KEY = "your_key_here"  # 직접 입력시 주석 해제

# API 키 상태 확인
api_keys_ready = True

if OPENAI_API_KEY:
    print("✅ OpenAI API 키가 설정되었습니다.")
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
else:
    print("❌ OPENAI_API_KEY가 설정되지 않았습니다!")
    print("📝 .env 파일에 OPENAI_API_KEY=your_key_here 를 추가하세요.")
    api_keys_ready = False

if SERPER_API_KEY:
    print("✅ Serper API 키가 설정되었습니다.")
    os.environ['SERPER_API_KEY'] = SERPER_API_KEY
else:
    print("❌ SERPER_API_KEY가 설정되지 않았습니다!")
    print("📝 https://serper.dev 에서 무료 키를 받아 설정하세요.")
    print("⚠️ 이 프로젝트는 Serper API 없이 실행할 수 없습니다.")
    api_keys_ready = False

if api_keys_ready:
    print("🎉 모든 API 키가 준비되었습니다!")
else:
    print("⚠️ API 키 설정을 완료한 후 진행하세요.")


✅ OpenAI API 키가 설정되었습니다.

✅ Serper API 키가 설정되었습니다.

🎉 모든 API 키가 준비되었습니다!

## 3. LLM 및 도구 설정

### LLM (Large Language Model) 설정

### CrewAI 도구 설정
- **SerperDevTool**: 웹 검색, 주식 데이터 수집, 뉴스 수집의 핵심 도구
- **FileWriterTool**: 분석 결과 파일 저장

### Serper API 활용 전략
- 다양한 검색 쿼리로 종합적 데이터 수집
- 실시간 주가, 재무 지표, 뉴스를 통합 검색
- 검색 결과의 신뢰성 검증 및 교차 확인


In [ ]:
# LLM 설정
if OPENAI_API_KEY:
    # OpenAI 모델 설정 (비용 효율적인 4o-mini 사용)
    llm = LLM(model="openai/gpt-4o-mini")
    print("✅ LLM 모델이 설정되었습니다.")

else:
    print("❌ OpenAI API 키가 없어 LLM을 설정할 수 없습니다.")
    llm = None

# 도구 설정
if SERPER_API_KEY:
    # Serper 검색 도구 (주식 데이터 및 뉴스 검색용)
    search_tool = SerperDevTool()
    print("✅ Serper 검색 도구가 활성화되었습니다.")

    # 파일 작성 도구
    file_writer = FileWriterTool()
    print("✅ 파일 작성 도구가 설정되었습니다.")

    print("🚀 모든 도구가 준비되었습니다!")
else:
    print("❌ Serper API 키가 없어 검색 도구를 설정할 수 없습니다.")
    search_tool = None
    file_writer = None

# 도구 준비 상태 확인
tools_ready = search_tool is not None and file_writer is not None and llm is not None

if tools_ready:
    print("🎯 시스템이 완전히 준비되었습니다!")
else:
    print("⚠️ 일부 도구가 설정되지 않았습니다. API 키를 확인하세요.")


✅ LLM 모델이 설정되었습니다.

✅ Serper 검색 도구가 활성화되었습니다.

✅ 파일 작성 도구가 설정되었습니다.

🚀 모든 도구가 준비되었습니다!

🎯 시스템이 완전히 준비되었습니다!

## 4. AI 에이전트 정의

투자 리서치팀을 구성하는 5명의 전문가 AI 에이전트를 정의합니다. 모든 에이전트가 Serper API를 활용하여 데이터를 수집하고 분석합니다.

### 에이전트 구성 (Serper API 특화)
1. **데이터 수집가 (Data Collector)**
   - Serper API를 통한 실시간 주가 및 재무 데이터 수집
   - 기업 정보 및 기본 통계 확보

2. **뉴스 분석가 (Financial News Analyst)**
   - 최신 뉴스 수집 및 감정 분석
   - 시장 심리 및 영향도 평가

3. **기술적 분석가 (Technical Analysis Expert)**  
   - 수집된 주가 데이터로 기술적 지표 계산
   - 차트 패턴 분석 및 매매 신호 생성

4. **기본 분석가 (Fundamental Analysis Specialist)**
   - 재무제표 데이터 분석
   - 기업 내재 가치 평가

5. **보고서 작성자 (Investment Report Writer)**
   - 종합 분석 결과 정리
   - 투자 의견 및 권고사항 제시

### Serper API 검색 전략
각 에이전트는 특화된 검색 쿼리를 사용하여 필요한 데이터를 수집합니다.


In [ ]:
# 도구가 준비된 경우에만 에이전트 생성
if tools_ready:
    # 1) 데이터 수집가 Agent
    data_collector = Agent(
        role="Stock Data Collector",
        goal="Serper API를 활용하여 주식의 실시간 가격, 재무 데이터, 기업 정보를 종합적으로 수집한다",
        backstory="""당신은 10년 경력의 금융 데이터 전문가입니다.
        다양한 온라인 소스에서 정확하고 신뢰할 수 있는 주식 데이터를 수집하는 능력으로 유명합니다.
        Yahoo Finance, Google Finance, Bloomberg 등의 데이터를 교차 검증하여
        가장 정확한 정보를 제공하는 전문성을 갖고 있습니다.
        특히 실시간 주가, PER, PBR, 시가총액 등 핵심 지표 수집에 뛰어납니다.""",
        tools=[search_tool],
        llm=llm,
        verbose=True, #중간 추론 과정을 보여 줄 것인지 확인
        allow_delegation=False
    )

    # 2) 뉴스 분석가 Agent
    news_analyst = Agent(
        role="Financial News Analyst",
        goal="최신 뉴스를 수집하고 감정 분석을 통해 시장 심리를 파악한다",
        backstory="""당신은 15년 경력의 금융 뉴스 전문 분석가입니다.
        Bloomberg, Reuters, CNBC, MarketWatch 등에서 근무한 경험이 있으며,
        뉴스가 주식 가격에 미치는 영향을 정확히 예측하는 능력으로 유명합니다.
        특히 뉴스의 감정적 톤과 시장 반응의 상관관계 분석에 전문성을 갖고 있으며,
        어닝스, 합병, 신제품 출시 등 주요 이벤트의 영향도를 정확히 평가합니다.""",
        tools=[search_tool],
        llm=llm,
        verbose=True,
        allow_delegation=False
    )

    print("✅ 데이터 수집가와 뉴스 분석가가 생성되었습니다.")
else:
    print("❌ 에이전트를 생성할 수 없습니다. API 키와 도구를 먼저 설정하세요.")


✅ 데이터 수집가와 뉴스 분석가가 생성되었습니다.

In [ ]:
if tools_ready:
    # 3) 기술적 분석가 Agent
    technical_analyst = Agent(
        role="Technical Analysis Expert",
        goal="수집된 주가 데이터를 분석하여 차트 패턴과 기술적 지표를 계산하고 매매 시점을 제안한다",
        backstory="""당신은 CFA, CMT 자격을 보유한 기술적 분석 전문가입니다.
        20년간 헤지펀드에서 퀀트 트레이딩 시스템을 개발하고 운영했으며,
        이동평균, RSI, MACD, 볼린저 밴드, 스토캐스틱 등 다양한 기술적 지표를 활용한
        정교한 차트 분석으로 높은 수익률을 달성한 경험이 있습니다.
        특히 지지선, 저항선 분석과 패턴 인식에 뛰어난 능력을 갖고 있습니다.""",
        tools=[search_tool, file_writer],
        llm=llm,
        verbose=True,
        allow_code_execution=False,
        allow_delegation=False
    )

    # 4) 기본 분석가 Agent
    fundamental_analyst = Agent(
        role="Fundamental Analysis Specialist",
        goal="기업의 재무제표와 비즈니스 모델을 분석하여 내재 가치를 평가한다",
        backstory="""당신은 워렌 버핏 스타일의 가치 투자 전문가입니다.
        Big 4 회계법인에서 15년간 기업 분석 업무를 담당했으며,
        PER, PBR, ROE, ROA, 부채비율, 유동비율 등 핵심 재무 지표를 통해
        기업의 내재 가치를 정확히 평가하는 능력을 보유하고 있습니다.
        특히 장기 투자 관점에서의 기업 성장성 분석과 경쟁 우위 평가에
        뛰어난 실력을 갖고 있습니다.""",
        tools=[search_tool, file_writer],
        llm=llm,
        verbose=True,
        allow_code_execution=False, # True라면 이 친구는 코드 인터프리터 도구 할당 받음 -> 코드를 작성하고 실행할 수 있는 권한
        allow_delegation=False
    )

    print("✅ 기술적 분석가와 기본 분석가가 생성되었습니다.")
else:
    print("❌ 에이전트를 생성할 수 없습니다.")


✅ 기술적 분석가와 기본 분석가가 생성되었습니다.

In [ ]:
if tools_ready:
    # 5) 종합 보고서 작성자 Agent
    report_writer = Agent(
        role="Investment Report Writer",
        goal="모든 분석 결과를 통합하여 투자자가 이해하기 쉬운 종합 보고서를 작성한다",
        backstory="""당신은 모건스탠리 리서치팀의 수석 애널리스트 출신으로,
        기관투자자와 개인투자자 모두에게 명확하고 실용적인 투자 보고서를
        작성하는 능력으로 정평이 나있습니다. 복잡한 분석 결과를
        이해하기 쉽게 정리하고, 명확한 투자 의견과 목표가격을 제시하는
        전문성을 보유하고 있습니다. 특히 리스크 요인을 명확히 제시하고
        다양한 시나리오 분석을 통해 투자자의 의사결정을 돕는 데 뛰어납니다.""",
        tools=[search_tool, file_writer],
        llm=llm,
        verbose=True,
        allow_delegation=True  # 다른 Agent들과 협업 가능
    )

    print("✅ 보고서 작성자가 생성되었습니다.")
    print("🎉 모든 AI 에이전트가 성공적으로 생성되었습니다!")

    # 에이전트 목록 확인
    agents_list = [data_collector, news_analyst, technical_analyst, fundamental_analyst, report_writer]
    print(f"📋 총 {len(agents_list)}명의 전문가 에이전트가 준비되었습니다:")
    for i, agent in enumerate(agents_list, 1):
        print(f"   {i}. {agent.role}")

else:
    print("❌ 보고서 작성자를 생성할 수 없습니다.")


✅ 보고서 작성자가 생성되었습니다.

🎉 모든 AI 에이전트가 성공적으로 생성되었습니다!

📋 총 5명의 전문가 에이전트가 준비되었습니다:

1. Stock Data Collector

2. Financial News Analyst

3. Technical Analysis Expert

4. Fundamental Analysis Specialist

5. Investment Report Writer

## 5. Task 정의 (Serper API 특화)

각 에이전트가 수행할 구체적인 작업들을 정의합니다. 모든 Task는 Serper API를 활용하여 데이터를 수집하고 분석합니다.

### Task 구성 (Serper API 기반)
1. **데이터 수집 Task**: Serper API로 실시간 주가, 재무 지표, 기업 정보 수집
2. **뉴스 분석 Task**: 최신 뉴스 수집 및 감정 분석
3. **기술적 분석 Task**: 수집된 주가 데이터로 기술적 지표 계산 및 차트 분석
4. **기본 분석 Task**: 재무 데이터로 기업 가치 및 성장성 분석
5. **종합 보고서 Task**: 모든 분석 결과 통합 및 투자 의견 제시

### Serper API 검색 쿼리 전략
- **주가 데이터**: "{stock_symbol} stock price today", "{stock_symbol} market cap"
- **재무 데이터**: "{stock_symbol} PE ratio PBR", "{company_name} financial statements"
- **뉴스 분석**: "{stock_symbol} news today", "{company_name} earnings"
- **경쟁 분석**: "{company_name} competitors", "{industry} sector analysis"

### Task 실행 순서
Tasks는 순차적으로 실행되며, 이전 Task의 결과를 다음 Task에서 참조할 수 있습니다.


In [ ]:
def create_serper_tasks(stock_symbol, company_name=""):
    """Serper API를 활용한 Task들을 생성하는 함수"""

    if not tools_ready:
        print("❌ 도구가 준비되지 않아 Task를 생성할 수 없습니다.")
        return []

    # Task 1: 데이터 수집
    data_collection_task = Task(
        description=f"""
        Serper API를 사용하여 {stock_symbol} ({company_name})에 대한 포괄적인 데이터를 수집하세요.

        수행할 검색 및 수집 작업:
        1. 실시간 주가 정보 검색:
           - "{stock_symbol} stock price today"
           - "{stock_symbol} current market cap"
           - "{stock_symbol} 52 week high low"

        2. 핵심 재무 지표 검색:
           - "{stock_symbol} PE ratio"
           - "{stock_symbol} PBR price to book"
           - "{stock_symbol} dividend yield"
           - "{company_name} revenue profit margin"

        3. 기업 기본 정보 검색:
           - "{company_name} business model"
           - "{company_name} CEO management team"
           - "{company_name} main products services"

        4. 시장 정보 수집:
           - "{company_name} market share"
           - "{company_name} competitors"
           - "{stock_symbol} trading volume"

        모든 검색 결과를 정리하여 다음 분석가들이 활용할 수 있도록 구조화된 데이터를 제공하세요.
        """,
        expected_output="실시간 주가, 재무 지표, 기업 정보, 시장 데이터를 포함한 종합적인 데이터 수집 보고서",
        agent=data_collector
    )

    return [data_collection_task]

print("✅ Task 생성 함수가 정의되었습니다 (데이터 수집).")


✅ Task 생성 함수가 정의되었습니다 (데이터 수집).

In [ ]:
def create_all_serper_tasks(stock_symbol, company_name=""):
    """Serper API를 활용한 모든 Task들을 생성하는 완전한 함수"""

    if not tools_ready:
        print("❌ 도구가 준비되지 않아 Task를 생성할 수 없습니다.")
        return []

    # Task 1: 데이터 수집
    data_collection_task = Task(
        description=f"""
        Serper API를 사용하여 {stock_symbol} ({company_name})에 대한 포괄적인 데이터를 수집하세요.
        다음 검색 쿼리들을 사용하여 정확하고 최신의 데이터를 수집하세요:

        주가 데이터: "{stock_symbol} stock price", "{stock_symbol} market cap"
        재무 지표: "{stock_symbol} PE ratio PBR", "{company_name} financial data"
        기업 정보: "{company_name} business overview", "{company_name} competitors"
        """,
        expected_output="실시간 주가, 재무 지표, 기업 정보를 포함한 종합적인 데이터 수집 보고서",
        agent=data_collector
    )

    # Task 2: 뉴스 분석
    news_analysis_task = Task(
        description=f"""
        Serper API를 사용하여 {stock_symbol} ({company_name})에 대한 최신 뉴스를 수집하고 분석하세요.

        수행할 검색 및 분석 작업:
        1. 최신 뉴스 검색:
           - "{stock_symbol} news today"
           - "{company_name} earnings news"
           - "{company_name} latest announcement"

        2. 감정 분석 수행:
           - 각 뉴스의 긍정/부정/중립 분류
           - 주요 이슈의 시장 영향도 평가
           - 전체적인 뉴스 심리 점수 산출 (1-10점)

        3. 중요 이벤트 식별:
           - 어닝스 발표, 신제품 출시, 인수합병 등
           - 규제 변화, 경쟁사 동향 등
        """,
        expected_output="뉴스 감정 분석 결과와 시장 영향도 평가를 포함한 뉴스 분석 보고서",
        agent=news_analyst,
        context=[data_collection_task]  # 데이터 수집 결과 참고
    )

    return [data_collection_task, news_analysis_task]

print("✅ 확장된 Task 생성 함수가 정의되었습니다 (데이터 수집 + 뉴스 분석).")


✅ 확장된 Task 생성 함수가 정의되었습니다 (데이터 수집 + 뉴스 분석).

In [ ]:
def create_complete_serper_tasks(stock_symbol, company_name=""):
    """Serper API를 활용한 완전한 5개 Task들을 생성하는 함수"""

    if not tools_ready:
        print("❌ 도구가 준비되지 않아 Task를 생성할 수 없습니다.")
        return []

    # 이전 Task들 (데이터 수집, 뉴스 분석)
    tasks = create_all_serper_tasks(stock_symbol, company_name)
    data_collection_task, news_analysis_task = tasks

    # Task 3: 기술적 분석
    technical_analysis_task = Task(
        description=f"""
        수집된 {stock_symbol}의 주가 데이터를 바탕으로 기술적 분석을 수행하세요.

        추가 데이터 검색 (필요시):
        - "{stock_symbol} historical stock price"
        - "{stock_symbol} technical analysis"
        - "{stock_symbol} chart pattern"

        수행할 분석:
        1. 기술적 지표 계산 및 해석:
           - 이동평균선 분석 (단기/중기/장기 추세)
           - RSI 분석 (과매수/과매도 구간)
           - MACD 분석 (매매 신호)
           - 볼린저 밴드 분석 (변동성 및 지지/저항선)

        2. 차트 패턴 분석:
           - 추세선 분석 (상승/하락/횡보)
           - 지지선과 저항선 식별
           - 패턴 인식 (삼각형, 쐐기형, 헤드앤숄더 등)

        3. 매매 신호 생성:
           - 단기/중기/장기 매매 신호
           - 목표가 및 손절가 제시
        """,
        expected_output="기술적 지표 분석 결과, 차트 패턴 분석, 매매 신호를 포함한 기술적 분석 보고서",
        agent=technical_analyst,
        context=[data_collection_task]  # 데이터 수집 결과 참고
    )

    # Task 4: 기본 분석
    fundamental_analysis_task = Task(
        description=f"""
        {stock_symbol} ({company_name})의 기본적 분석을 수행하세요.

        추가 재무 데이터 검색 (필요시):
        - "{company_name} financial statements 2024"
        - "{stock_symbol} earnings growth"
        - "{company_name} debt ratio"
        - "{company_name} industry comparison"

        수행할 분석:
        1. 재무 지표 분석:
           - 밸류에이션 지표 (PER, PBR, PEG)
           - 수익성 지표 (ROE, ROA, 영업이익률)
           - 안정성 지표 (부채비율, 유동비율)
           - 성장성 지표 (매출/순이익 증가율)

        2. 비즈니스 모델 분석:
           - 핵심 사업 경쟁력
           - 시장 지위 및 점유율
           - 성장 동력 및 미래 전망

        3. 동종업계 비교:
           - 주요 경쟁사 대비 밸류에이션
           - 업계 평균 대비 재무 성과

        4. 내재 가치 추정:
           - DCF 모델 기반 적정 주가
           - 멀티플 기반 목표 주가
        """,
        expected_output="재무 지표 분석, 비즈니스 모델 평가, 내재 가치 추정을 포함한 기본적 분석 보고서",
        agent=fundamental_analyst,
        context=[data_collection_task, technical_analysis_task]  # 이전 분석 결과들 참고
    )

    return [data_collection_task, news_analysis_task, technical_analysis_task, fundamental_analysis_task]

print("✅ 주요 4개 Task 생성 함수가 정의되었습니다.")


✅ 주요 4개 Task 생성 함수가 정의되었습니다.

In [ ]:
def create_final_serper_tasks(stock_symbol, company_name=""):
    """Serper API를 활용한 완전한 5개 Task들을 생성하는 최종 함수"""

    if not tools_ready:
        print("❌ 도구가 준비되지 않아 Task를 생성할 수 없습니다.")
        return []

    # 이전 4개 Task들
    tasks = create_complete_serper_tasks(stock_symbol, company_name)
    data_collection_task, news_analysis_task, technical_analysis_task, fundamental_analysis_task = tasks

    # Task 5: 종합 보고서 작성
    comprehensive_report_task = Task(
        description=f"""
        {stock_symbol} ({company_name})에 대한 종합 투자 분석 보고서를 작성하세요.

        추가 시장 동향 검색 (필요시):
        - "{company_name} analyst rating"
        - "{stock_symbol} price target"
        - "{company_name} industry outlook"
        - "{stock_symbol} investment recommendation"

        보고서에 포함할 내용:
        1. 임원 요약 (Executive Summary):
           - 투자 결론 및 추천 등급
           - 목표 주가 및 기간
           - 핵심 투자 포인트 3가지

        2. 데이터 수집 결과 요약:
           - 현재 주가 및 주요 재무 지표
           - 기업 개요 및 사업 모델
           - 시장 지위 및 경쟁 환경

        3. 뉴스 분석 결과 요약:
           - 최신 뉴스 감정 분석
           - 주요 이슈 및 이벤트 영향
           - 시장 심리 점수

        4. 기술적 분석 결과 요약:
           - 주요 기술적 지표 현황
           - 차트 패턴 및 추세 분석
           - 단기/중기 매매 신호

        5. 기본적 분석 결과 요약:
           - 핵심 재무 지표 평가
           - 동종업계 대비 경쟁력
           - 내재 가치 및 적정 주가

        6. 종합 투자 의견:
           - 투자 추천도 (강력매수/매수/보유/매도/강력매도)
           - 12개월 목표 주가
           - 투자 기간 및 전략 추천
           - 포트폴리오 비중 제안 (1-10%)

        7. 리스크 요인:
           - 주요 상승/하락 리스크
           - 시나리오별 주가 전망
           - 모니터링 포인트

        8. 투자 면책조항:
           - 교육/연구 목적임을 명시
           - 실제 투자 조언이 아님을 강조

        전문적이면서도 이해하기 쉬운 마크다운 형식으로 작성하세요. 한국어로 레포트를 작성하세요.
        """,
        expected_output="투자자를 위한 종합적이고 실용적인 투자 분석 보고서 (마크다운 형식)",
        agent=report_writer,
        context=[data_collection_task, news_analysis_task, technical_analysis_task, fundamental_analysis_task],
        output_file="serper_investment_report.md"  # 파일로 저장
    )

    return [data_collection_task, news_analysis_task, technical_analysis_task, fundamental_analysis_task, comprehensive_report_task]

print("✅ 완전한 5개 Task 생성 함수가 정의되었습니다!")


✅ 완전한 5개 Task 생성 함수가 정의되었습니다!

## 6. Crew 구성 및 실행 시스템 (Serper API 기반)

이제 정의된 에이전트들과 Task들을 하나의 팀(Crew)으로 구성하고 실행하는 시스템을 만듭니다.

### Crew 구성 요소 (Serper API 특화)
- **Agents**: 5명의 전문가 AI 에이전트 (모두 Serper API 활용)
- **Tasks**: 순차적으로 실행될 5개의 분석 작업
- **Process**: Sequential (순차적 실행)
- **Memory**: 에이전트 간 정보 공유 활성화

### 실행 프로세스 (Serper API 기반)
1. 주식 심볼 유효성 검증 (Serper API로 검색)
2. Task 생성 및 Crew 구성
3. 순차적 분석 실행:
   - 데이터 수집 (Serper API 검색)
   - 뉴스 분석 (Serper API 뉴스 검색)
   - 기술적 분석 (수집된 데이터 기반)
   - 기본 분석 (재무 데이터 분석)
   - 종합 보고서 (모든 결과 통합)
4. 최종 결과 파일 생성 (serper_investment_report.md)

### Serper API 활용의 장점
- 실시간 데이터 수집
- 다양한 소스의 정보 통합
- 뉴스 및 시장 동향 반영
- 경쟁사 및 업계 분석 포함


In [ ]:
def create_serper_investment_crew():
    """Serper API 기반 투자 리서치 크루를 생성하는 함수"""

    if not tools_ready:
        print("❌ 도구가 준비되지 않아 크루를 생성할 수 없습니다.")
        return None

    crew = Crew(
        agents=[
            data_collector,
            news_analyst,
            technical_analyst,
            fundamental_analyst,
            report_writer
        ],
        tasks=[],  # 나중에 동적으로 추가
        process=Process.sequential,  # 순차적 진행
        verbose=True,
        memory=True,  # 에이전트 간 정보 공유
    )

    return crew

def verify_stock_with_serper(stock_symbol, company_name=""):
    """Serper API를 사용하여 주식 심볼의 유효성을 검증하는 함수"""

    if not search_tool:
        print("❌ Serper API가 설정되지 않아 주식 검증을 할 수 없습니다.")
        return False

    try:
        # Serper API로 주식 정보 검색 시도
        print(f"🔍 Serper API로 {stock_symbol} 정보를 검증 중...")

        # 실제 검증은 Crew 실행 시 데이터 수집 단계에서 수행됩니다.
        # 여기서는 기본적인 형식 확인만 수행
        if len(stock_symbol) < 1 or len(stock_symbol) > 5:
            print(f"❌ {stock_symbol}은 유효하지 않은 주식 심볼 형식입니다.")
            return False

        print(f"✅ {stock_symbol} 형식이 유효합니다. Serper API로 상세 검증을 진행합니다.")
        return True

    except Exception as e:
        print(f"❌ 주식 검증 중 오류: {e}")
        return False

print("✅ Serper 기반 크루 생성 및 검증 함수가 정의되었습니다.")


✅ Serper 기반 크루 생성 및 검증 함수가 정의되었습니다.

In [ ]:
def analyze_stock_with_serper(stock_symbol, company_name=""):
    """Serper API를 활용하여 특정 주식에 대한 종합 분석을 수행하는 메인 함수"""

    print(f"🔍 {stock_symbol} ({company_name}) Serper API 기반 분석을 시작합니다...")
    print("=" * 70)

    # 1. 시스템 준비 상태 확인
    if not tools_ready:
        print("❌ 시스템이 준비되지 않았습니다!")
        print("📝 OpenAI API 키와 Serper API 키를 모두 설정한 후 다시 실행하세요.")
        return None

    # 2. 주식 심볼 기본 검증
    if not verify_stock_with_serper(stock_symbol, company_name):
        print(f"❌ {stock_symbol} 검증에 실패했습니다.")
        return None

    # 3. Task 생성
    print("📋 Serper API 기반 분석 Task들을 생성 중...")
    tasks = create_final_serper_tasks(stock_symbol, company_name)

    if not tasks:
        print("❌ Task 생성에 실패했습니다.")
        return None

    print(f"✅ {len(tasks)}개의 분석 Task가 생성되었습니다.")

    # 4. Crew 생성 및 Task 할당
    print("👥 투자 리서치 크루를 구성 중...")
    crew = create_serper_investment_crew()

    if not crew:
        print("❌ 크루 생성에 실패했습니다.")
        return None

    crew.tasks = tasks
    print("✅ 크루 구성이 완료되었습니다.")

    # 5. 분석 실행
    try:
        print(f"🚀 Serper API 기반 종합 분석을 실행합니다...")
        print("📊 예상 소요 시간: 8-15분 (Serper API 검색 포함)")
        print("💰 예상 비용: $1.00 - $3.00 (OpenAI API + Serper API)")
        print("-" * 70)

        result = crew.kickoff()

        print("\n" + "=" * 70)
        print("🎉 Serper API 기반 분석이 완료되었습니다!")
        print("📄 결과 파일: serper_investment_report.md")
        print("=" * 70)

        return result

    except Exception as e:
        print(f"❌ 분석 실행 중 오류 발생: {e}")
        print("💡 API 키 상태와 인터넷 연결을 확인해주세요.")
        return None

print("✅ Serper API 기반 주식 분석 시스템이 준비되었습니다!")


✅ Serper API 기반 주식 분석 시스템이 준비되었습니다!

## 7. 사용 예시 및 실행 (Serper API 버전)

이제 Serper API 기반 투자 분석 시스템이 완성되었습니다. 아래 셀에서 분석하고 싶은 주식을 설정하고 실행해보세요.

### 실행 전 체크리스트
1. **API 키 설정**: OpenAI API 키와 Serper API 키가 모두 올바르게 설정되었는지 확인
2. **인터넷 연결**: Serper API를 통한 데이터 수집을 위해 필요
3. **주식 심볼**: 분석하고 싶은 주식의 정확한 심볼 확인

### 추천 분석 대상 (Serper API 검증됨)
- **AAPL**: Apple Inc.
- **MSFT**: Microsoft Corporation  
- **GOOGL**: Alphabet Inc.
- **TSLA**: Tesla Inc.
- **NVDA**: NVIDIA Corporation
- **AMZN**: Amazon.com Inc.
- **META**: Meta Platforms Inc.

### Serper API 버전의 특징
- **실시간 데이터**: 최신 주가 및 뉴스 정보
- **종합적 검색**: 다양한 소스에서 데이터 수집
- **시장 동향**: 경쟁사 및 업계 분석 포함
- **뉴스 통합**: 최신 뉴스의 시장 영향도 분석

### 예상 소요 시간 및 비용
- **전체 분석**: 약 8-15분 (검색 시간 포함)
- **OpenAI API 비용**: 약 $1.00 - $2.50
- **Serper API 비용**: 무료 (월 2,500회 한도 내)


In [ ]:
# 실행 설정 - 여기서 분석할 주식을 설정하세요

# STOCK_SYMBOL = "AAPL"  # 분석할 주식 심볼
# COMPANY_NAME = "Apple Inc."  # 회사명

# 다른 예시들 (주석 해제하여 사용):
# STOCK_SYMBOL = "MSFT"
# COMPANY_NAME = "Microsoft Corporation"

# STOCK_SYMBOL = "GOOGLE"
# COMPANY_NAME = "Alphabet Inc."

STOCK_SYMBOL = "TSLA"
COMPANY_NAME = "Tesla Inc."

# STOCK_SYMBOL = "NVDA"
# COMPANY_NAME = "NVIDIA Corporation"

# STOCK_SYMBOL = "AMZN"
# COMPANY_NAME = "Amazon.com Inc."

# STOCK_SYMBOL = "META"
# COMPANY_NAME = "Meta Platforms Inc."

# STOCK_SYMBOL = "PKX"
# COMPANY_NAME = "POSCO HOLDINGS"

print(f"📊 분석 대상: {STOCK_SYMBOL} ({COMPANY_NAME})")
print("🔧 Serper API 기반 데이터 수집 모드")

if tools_ready:
    print("✅ 시스템 준비 완료! 다음 셀을 실행하여 분석을 시작하세요!")
else:
    print("❌ 시스템이 준비되지 않았습니다.")
    print("💡 위의 API 키 설정 셀에서 OpenAI API 키와 Serper API 키를 모두 설정하세요.")


📊 분석 대상: TSLA (Tesla Inc.)

🔧 Serper API 기반 데이터 수집 모드

✅ 시스템 준비 완료! 다음 셀을 실행하여 분석을 시작하세요!

In [ ]:
# Serper API 기반 종합 투자 분석 실행
def main_serper_analysis():
    """Serper API 기반 메인 실행 함수"""

    print("🏦 CrewAI + Serper API 종합 투자 리서치팀")
    print("=" * 60)
    print("⚠️  교육 목적 전용 - 실제 투자 조언 아님")
    print("🔍 Serper API 기반 실시간 데이터 수집")
    print("=" * 60)

    # API 키 확인
    if not api_keys_ready:
        print("❌ API 키가 완전히 설정되지 않았습니다!")
        print("📝 OpenAI API 키와 Serper API 키를 모두 설정한 후 다시 실행하세요.")
        print("\n📋 설정 방법:")
        print("1. .env 파일 생성")
        print("2. OPENAI_API_KEY=your_openai_key")
        print("3. SERPER_API_KEY=your_serper_key")
        return None

    # 분석 실행
    result = analyze_stock_with_serper(STOCK_SYMBOL, COMPANY_NAME)

    if result:
        print(f"\n📊 최종 결과:")
        print("-" * 50)
        print(result)

        # 결과 파일 확인
        if os.path.exists("serper_investment_report.md"):
            print(f"\n📁 상세 보고서가 'serper_investment_report.md' 파일에 저장되었습니다.")
            print("📈 보고서에는 다음 내용이 포함되어 있습니다:")
            print("   - Serper API 기반 실시간 데이터")
            print("   - 최신 뉴스 감정 분석")
            print("   - 기술적/기본적 분석")
            print("   - 종합 투자 의견")
        else:
            print(f"\n⚠️ 보고서 파일이 생성되지 않았습니다.")
            print("💡 분석 과정에서 오류가 발생했을 수 있습니다.")

    return result

In [ ]:
# 실행 (이 셀을 실행하면 Serper API 기반 분석이 시작됩니다)
if api_keys_ready and tools_ready:
    print("🚀 Serper API 기반 분석을 시작합니다...")
    print("⏱️ 예상 소요 시간: 3~5분")
    print("-" * 60)
    result = main_serper_analysis()
else:
    print("❌ 시스템이 준비되지 않았습니다.")
    print("📝 필수 요구사항:")
    print("   ✓ OpenAI API 키 설정")
    print("   ✓ Serper API 키 설정")
    print("   ✓ 인터넷 연결")
    print("\n💡 모든 요구사항을 충족한 후 다시 실행해주세요.")


🚀 Serper API 기반 분석을 시작합니다...

⏱️ 예상 소요 시간: 3~5분

------------------------------------------------------------

🏦 CrewAI + Serper API 종합 투자 리서치팀

============================================================

...

Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your execution traces?  (20s timeout): 
Would you like to view your exe

[CrewAIEventsBus] Sync handler error in on_llm_call_started: Only one live display may be active at once

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "TSLA PE ratio PBR"                                                                          │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {'searchParameters': {'q': 'TSLA PE ratio PBR', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic':   │
│  [{'title': 'Tesla, Inc. (TSLA) Valuation Measures & Financial Statistics', 'link':                             │
│  'https://finance.yahoo.com/quote/TSLA/key-statistics/', 'snippet': 'Find out all the key statistics for        │
│  Tesla, Inc. (TSLA) ... Trailing P/E, 259.52, 181.52, 127.04, 110.64, 73.49, 50.61. Forward P/E, 172.41,        │
│  163.93, 92.59, 117.65 ...', 'position': 1}, {'title': 'Tesla, Inc. (TSLA) Price/Earnings & PEG Ratios',        │
│  'link': 'https://www.nasdaq.com/market-activity/stocks/tsla/price-earnings-peg-ratios', 'snippet': 'Get        │
│  updated information on Tesla, Inc. Common Stock (TSLA) Price/Earnings Ratio (or PE Ratio) and PEG ratio for    │
│  stock evaluations with Nasdaq.', 'position': 2}, {'title': 'Tesla PE Ratio 2011-2025 | TSLA', 'link':          │
│  'https://www.macrotrends.net/stocks/charts/TSLA/tesla/pe-ratio', 'snippet': 'Tesla PE ratio as of October 20,  │
│  2025 is 258.42.\u200b\u200b Current and historical p/e ratio for Tesla (TSLA) from 2011 to 2025. The price to  │
│  earnings ratio is calculated ...', 'position': 3}, {'title': 'Tesla (TSLA) - P/B ratio', 'link':               │
│  'https://companiesmarketcap.com/tesla/pb-ratio/', 'snippet': "P/B ratio as of October 2025 : 18.7. According   │
│  to Tesla's latest financial reports the company has a price-to-book ratio of 15.7547. The price-to-book ratio  │
│  ...", 'position': 4}, {'title': 'Tesla (TSLA) - P/E ratio', 'link':                                            │
│  'https://companiesmarketcap.com/tesla/pe-ratio/', 'snippet': 'At the end of 2024 the company had a P/E ratio   │
│  of 181. P/E ratio history for Tesla from 2010 to 2025. 2013 2016 2019 2022 2025 ...', 'position': 5},          │
│  {'title': 'Tesla Price to Book Value Analysis', 'link':                                                        │
│  'https://ycharts.com/companies/TSLA/price_to_book_value', 'snippet': 'PE Ratio, 254.39. PS Ratio, 16.65.       │
│  Price to Free Cash Flow, 276.32. Price, 439.31. Earnings Yield, 0.39%. Market Cap, 1.417T. Operating PE        │
│  Ratio, 267.14.', 'position': 6}, {'title': 'Tesla, Inc. (TSLA) Stock Price, News, Quote & History',...         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 

In [ ]:
%cat serper_investment_report.md